In [0]:
dbutils.widgets.removeAll()

In [0]:
# DBTITLE 1,Create and Setup Parameters
# Define input widgets matching original Scala script
dbutils.widgets.text("FileId", "", "")
dbutils.widgets.text("FileLayoutDescription", "", "")
dbutils.widgets.text("CurrentContainer", "", "")
dbutils.widgets.text("CurrentFolderPath", "", "")
dbutils.widgets.text("ConversionType", "", "")
dbutils.widgets.text("FileName", "", "")         
dbutils.widgets.text("Delimiter", "", "")
dbutils.widgets.text("ConversionJsonContainer", "", "")
dbutils.widgets.text("ConversionJsonFolderPath", "", "")
dbutils.widgets.text("ConversionJsonFileName", "", "")
dbutils.widgets.text("HasControlFile", "", "")

In [0]:
file_id = dbutils.widgets.get("FileId")
file_layout_desc = dbutils.widgets.get("FileLayoutDescription")
current_container = dbutils.widgets.get("CurrentContainer")
current_folder_path = dbutils.widgets.get("CurrentFolderPath")
file_name = dbutils.widgets.get("FileName")
delimiter = dbutils.widgets.get("Delimiter")
conversion_type = dbutils.widgets.get("ConversionType")
conversion_json_container = dbutils.widgets.get("ConversionJsonContainer")
conversion_json_folder_path = dbutils.widgets.get("ConversionJsonFolderPath")
conversion_json_file_name = dbutils.widgets.get("ConversionJsonFileName")
has_control_file = dbutils.widgets.get("HasControlFile")

In [0]:
# Build target file and metadata path variables
mount_point = "/mnt/"
blob_path = f"{mount_point}{current_container}blob"
to_process_path = f"{blob_path}{current_folder_path}"
data_file_path = f"{to_process_path}/{file_name}"
conversion_json_path = f"{mount_point}{conversion_json_container}{conversion_json_folder_path}/{conversion_json_file_name}"

In [0]:
# Determine target path layout based on conversion type context
if conversion_type == "Generic":
    converted_file_path = f"{blob_path}/ConvertedFiles/{file_layout_desc}"
else:
    converted_file_path = f"{blob_path}/ConvertedFiles/{conversion_type}"

full_converted_file_path = f"{converted_file_path}/{file_name}"
control_file_name = f"{file_name}.ctl"
control_file_path = f"{converted_file_path}/{control_file_name}"
full_control_file_path = f"{control_file_path}/{control_file_name}"
ctl_process_path = f"{to_process_path}/{control_file_name}"

In [0]:
# DBTITLE 1,Conversion Type Routing Registry
notebook_registry = {
    "HICN_MBI": "../Conversion/HICNMBIConversion",
    "LISHIST": "../Conversion/LISHISTConversion",
    "MAO002": "../Conversion/MAO002Conversion",
    "MAO004": "../Conversion/MAO004Conversion",
    "MMR": "../Conversion/MMRConversion",
    "MOR": "../Conversion/MORConversion",
    "MORD": "../Conversion/MORDConversion",
    "PDERETURN": "../Conversion/PDEReturnConversion",
    "RAPS_RETURN": "../Conversion/RAPSReturnConversion",
    "MEMSD": "../Conversion/MEMSDConversion",
    "PPRD": "../Conversion/PPRDConversion",
    "DTRRD": "../Conversion/DTRRDConversion",
    "Generic": "../Conversion/GenericConversion",
    "OMIGPAL": "../Conversion/OMIGPALConversion",
    "HEDIS": "../Conversion/HEDISConversion",
    "Submitted837OutboundClaims": "../Conversion/Submitted837OutboundClaimsConversion"
}

In [0]:
# Find the matching target notebook path, default to 'Generic' if missing
notebook_path = notebook_registry.get(conversion_type, "../Conversion/GenericConversion")

In [0]:
# DBTITLE 1,Execute Target Notebook and Return Response
# Run the dynamically routed downstream validation/conversion notebook
return_json = dbutils.notebook.run(
    notebook_path,
    0,
    {
        "FileId": file_id,
        "FileLayoutDescription": file_layout_desc,
        "CurrentContainer": current_container,
        "CurrentFolderPath": current_folder_path,
        "FileName": file_name,
        "Delimiter": delimiter,
        "ConversionType": conversion_type,
        "ConversionJsonContainer": conversion_json_container,
        "ConversionJsonFolderPath": conversion_json_folder_path,
        "ConversionJsonFileName": conversion_json_file_name,
        "HasControlFile": has_control_file
    }
)

In [0]:
# Terminate session and pass structured string back to parent workflow caller
dbutils.notebook.exit(return_json)